# AAT CaT Spectrum Analysis for SP_Ace

This notebook is a student-facing workflow for one AAT CaT spectrum at a time.

**Goals**
1. Read in one spectrum.
2. Check the raw spectrum.
3. Estimate the S/N.
4. Normalize the spectrum.
5. Ask whether the normalization is acceptable.
6. If needed, use a restricted CaT-only pseudo-continuum normalization.
7. Automatically identify the CaT line centers.
8. Fit Voigt profiles to all three CaT lines and inspect the fits.
9. Measure EWs and lab-frame/raw RVs from the Voigt fitted centers.
10. Shift the normalized spectrum into the stellar rest/lab wavelength frame for SP_Ace.
11. Write the SP_Ace input file.
12. Append a compact results summary table.

**Important scientific rule:** do not blindly trust the algorithm. Inspect every plot.

In [ ]:
# ============================================================
# 0. Notebook display and plotting setup
# ============================================================

from IPython.display import display, HTML

display(HTML("""
<style>
.container { width: 100% !important; }
.output_area img { max-width: 100% !important; height: auto !important; }
.output_wrapper, .output { height:auto !important; max-height:2000px; }
.output_scroll { height:auto !important; box-shadow:none !important; }
</style>
"""))

%matplotlib inline

from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('retina')

import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.io import fits, ascii
from astropy.table import Table
from specutils import Spectrum1D

plt.rcParams.update({
    "figure.figsize": (16, 6),
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "legend.fontsize": 11,
    "lines.linewidth": 1.2,
    "figure.autolayout": True,
    "grid.alpha": 0.4,
})

print("Notebook display settings loaded.")


In [ ]:
# ============================================================
# 1. Clear old star variables before starting a new spectrum
# ============================================================
# Run this cell before processing each new star.

variables_to_clear = [
    # File/star labels
    "spec_path", "file_number", "fiberid", "star_id",

    # Spectrum objects and arrays
    "spectrum", "header", "raw_flux", "wave_AA", "wave_m",
    "x_m", "x_um", "wavelength_m", "normalized_flux_array", "flux_norm", "spec_norm", "spdata", "spec1",

    # Continuum variables
    "continuum", "continuum_fit", "norm_trim", "run_restricted_fit",

    # CaT centers and fits
    "c1", "c2", "c3", "picked_cat", "picked_cat_AA", "fitted_cat_AA",
    "center_results", "fit_results",

    # EW and RV products
    "ew_results", "rv_results", "dl_results", "cat1", "cat2", "cat3",
    "rv1_lab", "rv2_lab", "rv3_lab", "avrv", "sigrv",
    "SEW", "SEW2", "RV_barycentric", "RV_apogee_fit", "rverr",

    # SP_Ace export variables
    "wvl_rest_AA", "wvl_rest_m", "rv_shift", "outfile", "data",
]

for var in variables_to_clear:
    if var in globals():
        del globals()[var]

print("Old star variables cleared.")


In [ ]:
# ============================================================
# 2. List n6569_2_*.fits files in numerical order and choose one
# ============================================================

from dotenv import load_dotenv
import os

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "README.md").exists() and (Path.cwd().parent / "README.md").exists():
    PROJECT_ROOT = Path.cwd().parent.resolve()
load_dotenv(PROJECT_ROOT / ".env", override=True)

spec_dir = Path(os.getenv("RAW_FITS_DIR", PROJECT_ROOT / "data/raw/fits")).resolve()

fits_files = sorted(
    spec_dir.glob("n6569_2_*.fits"),
    key=lambda f: int(f.stem.split("_")[-1])
)

if len(fits_files) == 0:
    raise FileNotFoundError(f"No n6569_2_*.fits files found in {spec_dir}")

print("Available n6569_2 FITS files:\n")

entries = []
for i, f in enumerate(fits_files):
    fiber = int(f.stem.split("_")[-1])
    entries.append(f"[{i:3d}] fiber {fiber:4d}")

ncols = 6
nrows = math.ceil(len(entries) / ncols)
for r in range(nrows):
    row_text = ""
    for c in range(ncols):
        idx = r + c * nrows
        if idx < len(entries):
            row_text += f"{entries[idx]:<24}"
    print(row_text)

file_index = int(input("\nEnter file number from the bracketed list: "))
spec_path = str(fits_files[file_index])

fname = Path(spec_path).stem
file_number = fname.split("_")[-1]
fiberid = file_number
star_id = f"n6569_{file_number}"

print("\nSelected file:")
print(spec_path)
print(f"fiberid = {fiberid}")
print(f"star_id = {star_id}")


In [ ]:
# ============================================================
# 3. Read the FITS spectrum and make a Spectrum1D object
# ============================================================

with fits.open(spec_path) as hdul:
    header = hdul[0].header
    raw_data = np.asarray(hdul[0].data, dtype=float).squeeze()

# If the FITS file has more than one row/extension-like array, use the first 1D spectrum.
if raw_data.ndim > 1:
    raw_flux = np.asarray(raw_data[0], dtype=float).ravel()
else:
    raw_flux = np.asarray(raw_data, dtype=float).ravel()

npix = len(raw_flux)

# Build wavelength solution from the FITS WCS keywords.
# This assumes a linear wavelength axis.
crval = header.get("CRVAL1")
cdelt = header.get("CDELT1", header.get("CD1_1"))
crpix = header.get("CRPIX1", 1.0)
cunit = str(header.get("CUNIT1", "Angstrom")).lower()

if crval is None or cdelt is None:
    raise ValueError("Could not find CRVAL1/CDELT1 wavelength solution in FITS header.")

pix = np.arange(npix) + 1.0
wave_native = crval + (pix - crpix) * cdelt

# Guess/assign wavelength units.
if "angstrom" in cunit or "ang" in cunit:
    wave_q = wave_native * u.AA
elif cunit in ["nm", "nanometer", "nanometers"]:
    wave_q = wave_native * u.nm
elif cunit in ["m", "meter", "meters"]:
    wave_q = wave_native * u.m
else:
    med = np.nanmedian(wave_native)
    if med > 1000:
        wave_q = wave_native * u.AA
    elif 100 < med < 1000:
        wave_q = wave_native * u.nm
    else:
        wave_q = wave_native * u.m

wave_AA = wave_q.to_value(u.AA)
wave_m = wave_q.to_value(u.m)

spectrum = Spectrum1D(
    spectral_axis=wave_q.to(u.m),
    flux=raw_flux * u.dimensionless_unscaled
)

print("Spectrum read successfully.")
print(f"File: {spec_path}")
print(f"Number of pixels: {npix}")
print(f"Wavelength range: {wave_AA[0]:.2f}--{wave_AA[-1]:.2f} Angstrom")
print(f"Flux median: {np.nanmedian(raw_flux):.3f}")


In [ ]:
# ============================================================
# 4. Check the input spectrum visually
# ============================================================

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(wave_m, raw_flux, lw=1, label="Input spectrum")

# CaT rest wavelengths
for lam in [8498.02, 8542.09, 8662.14]:
    ax.axvline(lam * 1e-10, color="red", ls=":", alpha=0.8)

ax.set_xlim(8.39e-7, 8.89e-7)
ax.set_xlabel("Wavelength (m)")
ax.set_ylabel("Flux")
ax.set_title(f"Input AAT spectrum: {star_id}")
ax.grid(True)
ax.legend()
plt.show()


In [ ]:
# ============================================================
# 5. Estimate S/N near the CaT region
# ============================================================

def robust_sigma(values):
    """Robust scatter estimate using MAD."""
    values = np.asarray(values, dtype=float)
    med = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - med))
    return 1.4826 * mad

# Use relatively line-free continuum/pseudo-continuum windows.
snr_windows_AA = [
    (8510, 8530),
    (8560, 8620),
    (8690, 8780),
    (8750, 8790),
]

snr_mask = np.zeros_like(wave_AA, dtype=bool)
for lo, hi in snr_windows_AA:
    snr_mask |= (wave_AA > lo) & (wave_AA < hi)

# Fallback if too few points are available.
if np.sum(snr_mask) < 20:
    snr_mask = (wave_AA > 8450) & (wave_AA < 8800)

snr_flux = raw_flux[snr_mask]
med_flux = np.nanmedian(snr_flux)
sig_flux = robust_sigma(snr_flux)

snr = med_flux / sig_flux if sig_flux > 0 else np.nan

print("S/N estimate")
print("------------")
print(f"Median continuum/pseudo-continuum flux = {med_flux:.3f}")
print(f"Robust scatter = {sig_flux:.3f}")
print(f"S/N estimate = {snr:.1f}")


In [ ]:
# ============================================================
# 6. Full-spectrum continuum normalization
# ============================================================
# This works for normal stars. For cool AGB/M-type stars or bad fits,
# the next cells let students switch to a restricted CaT-only pseudo-continuum.

from astropy.modeling.polynomial import Chebyshev1D
from astropy.modeling.fitting import LevMarLSQFitter

x_um = spectrum.spectral_axis.to_value(u.um)
x_m = spectrum.spectral_axis.to_value(u.m)
y = spectrum.flux.value

sigma_frac = 1.0 / snr if np.isfinite(snr) and snr > 0 else 0.03

# Continuum windows chosen to avoid the CaT cores and unstable red edge.
cont_windows = (
    ((x_um > 0.8400) & (x_um < 0.8480)) |
    ((x_um > 0.8510) & (x_um < 0.8530)) |
    ((x_um > 0.8570) & (x_um < 0.8640)) |
    ((x_um > 0.8690) & (x_um < 0.8780)) |
    ((x_um > 0.8820) & (x_um < 0.8840)) |
    ((x_um > 0.8845) & (x_um < 0.8875))
)

# Remove bad pixels.
cont_windows &= np.isfinite(x_um) & np.isfinite(y) & (y > 0)

fitter = LevMarLSQFitter()
model0 = Chebyshev1D(4)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    g0_fit = fitter(model0, x_um[cont_windows], y[cont_windows])

cont0 = g0_fit(x_um)
norm0 = y / cont0

# S/N-based clipping in the continuum windows.
# Keep high continuum points but reject deep absorption and major spikes.
good_cont = cont_windows.copy()
good_cont &= (norm0 > 1 - 3 * sigma_frac)
good_cont &= (norm0 < 1 + 5 * sigma_frac)
good_cont &= np.isfinite(norm0)

model1 = Chebyshev1D(4)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    continuum_fit = fitter(model1, x_um[good_cont], y[good_cont])

continuum = continuum_fit(x_um)
normalized_flux_array = y / continuum
flux_norm = normalized_flux_array
spec_norm = normalized_flux_array
wavelength_m = x_m

spdata = Spectrum1D(
    spectral_axis=x_m * u.m,
    flux=normalized_flux_array * u.dimensionless_unscaled
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

ax1.plot(x_m, y, lw=1, label="Spectrum")
ax1.plot(x_m, continuum, lw=2, label="Continuum fit")
ax1.plot(x_m[good_cont], y[good_cont], ".", ms=2, alpha=0.5, label="Continuum points")
ax1.set_ylabel("Flux")
ax1.set_title("Full-spectrum continuum fit + normalized spectrum")
ax1.grid(True)
ax1.legend()

ax2.plot(x_m, normalized_flux_array, lw=1, label="Normalized spectrum")
ax2.axhline(1.0, color="black", ls="--")
ax2.set_ylim(0, 1.5)
ax2.set_xlim(8.395e-7, 8.89e-7)
ax2.set_xlabel("Wavelength (m)")
ax2.set_ylabel("Normalized Flux")
ax2.grid(True)
ax2.legend()

plt.show()

print("Full-spectrum normalization complete.")
print(f"Median normalized flux = {np.nanmedian(normalized_flux_array):.3f}")
print(spdata)


In [ ]:
# ============================================================
# 7. Student quality check: is the normalization acceptable?
# ============================================================
# Students must inspect the plots before continuing.

print()
print("====================================================")
print("CONTINUUM NORMALIZATION QUALITY CHECK")
print("====================================================")
print()
print("Inspect the plots carefully.")
print()
print("Good normalization:")

print("  - continuum near 1.0")
print("  - CaT lines centered cleanly")
print("  - no large slopes or spline failures")
print("  - no exploding red end")
print()
print("Bad normalization examples:")
print("  - continuum tilted strongly")
print("  - spectrum above/below 1.0")
print("  - spline diverges at red edge")
print("  - strong bumps/wiggles")
print("  - cool M/AGB star pseudo-continuum")
print()

response = input("Is the normalization acceptable? (y/n): ").strip().lower()

if response in ["y", "yes"]:
    run_restricted_fit = False
    print("\nUsing FULL-spectrum normalization. Continue to CaT line fitting.")
elif response in ["n", "no"]:
    run_restricted_fit = True
    print("\nUsing RESTRICTED CaT-only normalization. Run the next cell.")
else:
    run_restricted_fit = False
    print("\nInput not recognized. Defaulting to FULL-spectrum normalization.")

print(f"\nrun_restricted_fit = {run_restricted_fit}")


In [ ]:
# ============================================================
# 8. Optional restricted CaT-only pseudo-continuum normalization
# ============================================================
# Use this only when the full-spectrum normalization is bad.
# It fills the rest of the spectrum with 1.000 so later cells have stable arrays.

if not run_restricted_fit:

    print("Restricted fit skipped. Keeping previous full-spectrum normalization.")

else:

    print("Running restricted CaT-only fit...")

    from scipy.interpolate import UnivariateSpline
    from scipy.signal import medfilt

    wave = spectrum.spectral_axis
    flux = spectrum.flux

    x_um_all = wave.to_value(u.um)
    x_m_full = wave.to_value(u.m)
    y_full = flux.value

    # Do not trust the continuum fit beyond this red edge.
    red_stop_um = 0.8795

    good0 = (
        np.isfinite(x_um_all) &
        np.isfinite(y_full) &
        (y_full > 0) &
        (x_um_all > 0.848) &
        (x_um_all < red_stop_um)
    )

    x_um0 = x_um_all[good0]
    x_m0 = x_m_full[good0]
    y0 = y_full[good0]

    mask = np.ones_like(x_um0, dtype=bool)

    exclude_regions = [
        (0.8490, 0.8510),   # CaT 8498
        (0.8535, 0.8552),   # CaT 8542
        (0.8655, 0.8672),   # CaT 8662
    ]

    for lo, hi in exclude_regions:
        mask &= ~((x_um0 > lo) & (x_um0 < hi))

    y_med = medfilt(y0, kernel_size=31)
    ratio = y0 / y_med

    cont_points = (
        mask &
        np.isfinite(ratio) &
        (ratio > 0.95) &
        (ratio < 1.20)
    )

    anchor_windows = (
        ((x_um0 > 0.8510) & (x_um0 < 0.8530)) |
        ((x_um0 > 0.8560) & (x_um0 < 0.8620)) |
        ((x_um0 > 0.8690) & (x_um0 < 0.8740)) |
        ((x_um0 > 0.8750) & (x_um0 < 0.8790))
    )

    cont_points |= anchor_windows & mask & np.isfinite(y0)

    order = np.argsort(x_um0)
    xfit = x_um0[order]
    yfit = y0[order]
    fitmask = cont_points[order]

    if np.sum(fitmask) < 20:
        raise ValueError(
            "Too few continuum points for spline fit. Loosen ratio cut or add anchors."
        )

    s_value = len(xfit[fitmask]) * np.nanvar(yfit[fitmask]) * 0.30

    spline = UnivariateSpline(
        xfit[fitmask],
        yfit[fitmask],
        k=3,
        s=s_value
    )

    continuum0 = spline(x_um0)

    ok_cont = (
        np.isfinite(continuum0) &
        (continuum0 > 0) &
        (x_um0 <= red_stop_um)
    )

    x_m_norm = x_m0[ok_cont]
    x_um_norm = x_um0[ok_cont]
    continuum = continuum0[ok_cont]
    norm_trim = y0[ok_cont] / continuum

    wave_full = spectrum.spectral_axis.to_value(u.m)
    wave_full_um = spectrum.spectral_axis.to_value(u.um)

    norm_full = np.ones_like(wave_full, dtype=float)

    inside = (
        (wave_full >= x_m_norm[0]) &
        (wave_full <= x_m_norm[-1]) &
        (wave_full_um <= red_stop_um)
    )

    norm_full[inside] = np.interp(
        wave_full[inside],
        x_m_norm,
        norm_trim
    )

    # Force unusable red end back to unity.
    norm_full[wave_full_um > red_stop_um] = 1.0

    x_m = wave_full
    x_um = wave_full_um
    wavelength_m = wave_full
    normalized_flux_array = norm_full
    flux_norm = norm_full
    spec_norm = norm_full

    good = np.isfinite(x_m) & np.isfinite(normalized_flux_array)

    spdata = Spectrum1D(
        spectral_axis=x_m[good] * u.m,
        flux=normalized_flux_array[good] * u.dimensionless_unscaled
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 9), sharex=True)

    ax1.plot(x_m0, y0, lw=1, label="Spectrum")
    ax1.plot(x_m0[cont_points], y0[cont_points], ".", ms=2, alpha=0.6, label="Continuum points")
    ax1.plot(x_m_norm, continuum, lw=2, label="Spline continuum")
    ax1.axvline(red_stop_um * 1e-6, color="red", ls=":", label="Red cutoff")
    ax1.set_ylabel("Flux")
    ax1.set_title("Restricted CaT-only spline continuum fit")
    ax1.grid(True)
    ax1.legend()

    ax2.plot(x_m_norm, norm_trim, lw=1, label="Normalized restricted region")
    ax2.axhline(1.0, color="black", ls="--")
    ax2.axvline(red_stop_um * 1e-6, color="red", ls=":", label="Red cutoff")
    ax2.set_ylim(0, 1.5)
    ax2.set_xlim(8.48e-7, 8.825e-7)
    ax2.set_xlabel("Wavelength (m)")
    ax2.set_ylabel("Normalized Flux")
    ax2.grid(True)
    ax2.legend()

    plt.show()

    fig11, ax11 = plt.subplots(figsize=(14, 5))
    ax11.plot(x_m, normalized_flux_array, label="Full padded normalized spectrum")
    ax11.axhline(1.0, color="black", ls="--")
    ax11.axvline(red_stop_um * 1e-6, color="red", ls=":", label="Red cutoff")
    ax11.set_ylim(0, 1.5)
    ax11.set_xlabel("Wavelength (m)")
    ax11.set_ylabel("Normalized Flux")
    ax11.grid(True)
    ax11.legend(loc="lower left", frameon=True)
    plt.show()

    print("CaT-only normalization complete.")
    print(f"Median normalized flux in restricted region = {np.nanmedian(norm_trim):.3f}")
    print(f"Restricted wavelength range = {x_m_norm[0]:.3e} to {x_m_norm[-1]:.3e} m")
    print(f"Red cutoff = {red_stop_um:.4f} um")
    print(f"Full wavelength length = {len(x_m)}")
    print(f"Full normalized flux length = {len(normalized_flux_array)}")
    print(spdata)


In [ ]:
# ============================================================
# 9. Find initial CaT line centers automatically
# ============================================================
# These are ONLY starting guesses for the Voigt fits.
# We first find a common RV shift for the CaT triplet pattern,
# then search near the shifted expected line positions.

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

rest_cat_m = np.array([8.49802e-07, 8.54209e-07, 8.66214e-07])
rest_cat_AA = rest_cat_m * 1.0e10
cat_names = ["CaT1 8498", "CaT2 8542", "CaT3 8662"]

c_kms = 299792.458

r_m = np.asarray(x_m, dtype=float)
spec_norm = np.asarray(normalized_flux_array, dtype=float)

good = np.isfinite(r_m) & np.isfinite(spec_norm) & (spec_norm > 0)
r_m = r_m[good]
spec_norm = spec_norm[good]

order = np.argsort(r_m)
r_m = r_m[order]
spec_norm = spec_norm[order]

r_AA = r_m * 1.0e10

# Smooth lightly so we do not pick individual noisy pixels
smooth_flux = median_filter(spec_norm, size=5)

# ------------------------------------------------------------
# Step 1: find approximate RV by scanning the CaT triplet pattern
# ------------------------------------------------------------

rv_grid = np.linspace(-250, 250, 1001)  # km/s
scores = []

for rv_try in rv_grid:

    shifted_cat_AA = rest_cat_AA * (1.0 + rv_try / c_kms)

    score = 0.0

    for center in shifted_cat_AA:

        # Look near expected shifted line center
        m = (
            (r_AA > center - 2.5) &
            (r_AA < center + 2.5)
        )

        if np.sum(m) > 3:
            # Strong absorption gives large 1 - flux
            score += np.nanmax(1.0 - smooth_flux[m])

    scores.append(score)

scores = np.array(scores)

rv_guess_kms = rv_grid[np.nanargmax(scores)]

expected_cat_AA = rest_cat_AA * (1.0 + rv_guess_kms / c_kms)

print("Best automatic CaT triplet RV guess:")
print(f"  RV_guess = {rv_guess_kms:.2f} km/s")
print()

# ------------------------------------------------------------
# Step 2: find local minima near shifted expected CaT positions
# ------------------------------------------------------------

search_half_width_AA = 2.5
picked_cat_AA = []

for name, center_guess in zip(cat_names, expected_cat_AA):

    window = (
        (r_AA > center_guess - search_half_width_AA) &
        (r_AA < center_guess + search_half_width_AA)
    )

    if np.sum(window) < 5:
        raise ValueError(f"Not enough pixels in search window for {name}.")

    ww = r_AA[window]
    ff = smooth_flux[window]

    i_min = np.argmin(ff)

    # Optional small parabolic refinement around the minimum
    if 0 < i_min < len(ww) - 1:
        x3 = ww[i_min-1:i_min+2]
        y3 = ff[i_min-1:i_min+2]

        try:
            a, b, c = np.polyfit(x3, y3, 2)

            if a > 0:
                center_refined = -b / (2*a)

                if ww.min() <= center_refined <= ww.max():
                    picked_cat_AA.append(center_refined)
                else:
                    picked_cat_AA.append(ww[i_min])
            else:
                picked_cat_AA.append(ww[i_min])

        except Exception:
            picked_cat_AA.append(ww[i_min])

    else:
        picked_cat_AA.append(ww[i_min])

picked_cat_AA = np.array(picked_cat_AA)
picked_cat = picked_cat_AA * 1.0e-10

c1, c2, c3 = picked_cat

rv_initial = c_kms * (picked_cat_AA - rest_cat_AA) / rest_cat_AA

print("Initial automatic CaT centers, for Voigt starting guesses only:")
for name, rest, expected, obs, rv in zip(
    cat_names,
    rest_cat_AA,
    expected_cat_AA,
    picked_cat_AA,
    rv_initial
):
    print(
        f"{name}: rest={rest:.2f} Å, "
        f"expected={expected:.3f} Å, "
        f"initial={obs:.3f} Å, "
        f"initial RV={rv:.2f} km/s"
    )

# ------------------------------------------------------------
# Plot sanity check
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(r_m, spec_norm, lw=1, label="Normalized spectrum")

for c in picked_cat:
    ax.axvline(
        c,
        color="black",
        ls=":",
        lw=2,
        label="Initial center" if c == picked_cat[0] else None
    )

for rc in rest_cat_m:
    ax.axvline(
        rc,
        color="red",
        ls="--",
        alpha=0.8,
        label="Rest CaT" if rc == rest_cat_m[0] else None
    )

for ec in expected_cat_AA * 1.0e-10:
    ax.axvline(
        ec,
        color="purple",
        ls="-.",
        alpha=0.8,
        label="Expected shifted CaT" if ec == expected_cat_AA[0] * 1.0e-10 else None
    )

ax.axhline(1.0, color="gray", ls="--")

ax.set_xlim(8.47e-7, 8.70e-7)
ax.set_ylim(0, 1.4)

ax.set_xlabel("Wavelength (m)")
ax.set_ylabel("Normalized Flux")
ax.set_title("Initial automatic CaT centers using common triplet RV shift")

ax.grid(True)
ax.legend()

plt.show()

In [ ]:
# ============================================================
# Robust CaT RV from simultaneous 3-line Voigt fit
# Allows RV search over +/- 500 km/s
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

from scipy.optimize import curve_fit
from scipy.special import voigt_profile

# ------------------------------------------------------------
# Constants and line information
# ------------------------------------------------------------

c_kms = 299792.458

rest_cat_AA = np.array([8498.02, 8542.09, 8662.14], dtype=float)

cat_names = ["CaT1", "CaT2", "CaT3"]

# ------------------------------------------------------------
# Current normalized spectrum
# ------------------------------------------------------------

lam_AA = np.asarray(x_m, dtype=float) * 1.0e10
flux = np.asarray(normalized_flux_array, dtype=float)

good = (
    np.isfinite(lam_AA) &
    np.isfinite(flux) &
    (flux > 0)
)

lam_AA = lam_AA[good]
flux = flux[good]

# ------------------------------------------------------------
# RV search settings
# ------------------------------------------------------------

rv_guess = avrv.to_value(u.km/u.s) if "avrv" in globals() else 0.0

rv_search_width = 500.0  # km/s

# Convert +/- RV range into wavelength padding
rv_pad_AA = rest_cat_AA * rv_search_width / c_kms

# Extra local half-width for line shape
line_half_width_AA = np.array([7.0, 9.0, 9.0], dtype=float)

# Total fitting window around each expected line
fit_half_width = rv_pad_AA + line_half_width_AA

# Expected centers from initial RV guess
shifted_guess = rest_cat_AA * (1.0 + rv_guess / c_kms)

# ------------------------------------------------------------
# Build mask around all possible shifted CaT positions
# ------------------------------------------------------------

mask = np.zeros_like(lam_AA, dtype=bool)

for center, hw in zip(shifted_guess, fit_half_width):

    mask |= (
        (lam_AA > center - hw) &
        (lam_AA < center + hw)
    )

xfit = lam_AA[mask]
yfit = flux[mask]

if len(xfit) < 50:
    raise ValueError("Too few points for simultaneous CaT RV fit.")

print(f"Initial RV guess = {rv_guess:.2f} km/s")
print(f"RV search range  = +/- {rv_search_width:.1f} km/s")
print(f"Pixels used      = {len(xfit)}")

# ------------------------------------------------------------
# Three-line shared-RV model
# ------------------------------------------------------------

def three_cat_model(
    lam,
    rv,
    cont0,
    slope,
    d1,
    sig1,
    gam1,
    d2,
    sig2,
    gam2,
    d3,
    sig3,
    gam3
):
    """
    Three Ca II triplet Voigt absorption lines with one shared RV.
    Each line has its own depth and width.
    """

    continuum = cont0 + slope * (lam - 8580.0)

    model = np.ones_like(lam)

    depths = [d1, d2, d3]
    sigmas = [sig1, sig2, sig3]
    gammas = [gam1, gam2, gam3]

    centers = rest_cat_AA * (1.0 + rv / c_kms)

    for center, depth, sigma, gamma in zip(
        centers,
        depths,
        sigmas,
        gammas
    ):

        prof = voigt_profile(
            lam - center,
            sigma,
            gamma
        )

        prof /= voigt_profile(
            0.0,
            sigma,
            gamma
        )

        model *= (
            1.0 - depth * prof
        )

    return continuum * model

# ------------------------------------------------------------
# Initial guesses
# ------------------------------------------------------------

p0 = [
    rv_guess,   # common RV
    1.0,        # continuum level
    0.0,        # continuum slope

    0.25, 0.8, 0.8,   # CaT1 depth, sigma, gamma
    0.45, 1.0, 1.0,   # CaT2 depth, sigma, gamma
    0.40, 1.0, 1.0    # CaT3 depth, sigma, gamma
]

# ------------------------------------------------------------
# Bounds
# ------------------------------------------------------------

lower = [
    rv_guess - rv_search_width,
    0.6,
    -0.01,

    0.00, 0.05, 0.01,
    0.00, 0.05, 0.01,
    0.00, 0.05, 0.01
]

upper = [
    rv_guess + rv_search_width,
    1.4,
    0.01,

    0.95, 4.0, 6.0,
    0.95, 4.0, 6.0,
    0.95, 4.0, 6.0
]

# ------------------------------------------------------------
# Fit
# ------------------------------------------------------------

try:

    popt, pcov = curve_fit(
        three_cat_model,
        xfit,
        yfit,
        p0=p0,
        bounds=(lower, upper),
        maxfev=20000
    )

except RuntimeError:

    raise RuntimeError(
        "Simultaneous CaT RV fit failed to converge."
    )

# ------------------------------------------------------------
# Extract RV
# ------------------------------------------------------------

rv_simul = popt[0] * u.km / u.s

avrv = rv_simul

try:

    sigrv = np.sqrt(pcov[0, 0]) * u.km / u.s

except Exception:

    sigrv = np.nan * u.km / u.s

# Save simultaneous RV solution separately
avrv_simul = avrv
sigrv_simul = sigrv

print()
print(f"Simultaneous CaT RV = {avrv_simul:.3f} +/- {sigrv_simul:.3f}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

xplot = np.linspace(
    xfit.min(),
    xfit.max(),
    5000
)

ymodel = three_cat_model(
    xplot,
    *popt
)

fig, ax = plt.subplots(
    figsize=(16, 5)
)

ax.plot(
    xfit,
    yfit,
    "k.",
    ms=3,
    alpha=0.7,
    label="Data"
)

ax.plot(
    xplot,
    ymodel,
    "b-",
    lw=2,
    label="3-line shared-RV fit"
)

for i, rc in enumerate(rest_cat_AA):

    ax.axvline(
        rc,
        color="red",
        ls=":",
        label="Lab CaT" if i == 0 else None
    )

    ax.axvline(
        rc * (1.0 + avrv_simul.to_value(u.km/u.s) / c_kms),
        color="purple",
        ls="--",
        label="Fitted shifted CaT" if i == 0 else None
    )

ax.set_xlabel("Wavelength (Å)")
ax.set_ylabel("Normalized Flux")

ax.set_title("Simultaneous CaT RV Fit")

ax.set_ylim(0, 1.4)

ax.grid(True)

ax.legend()

plt.show()

# ------------------------------------------------------------
# Final printout
# ------------------------------------------------------------

print()
print("Saved simultaneous RV solution:")
print(f"  avrv_simul  = {avrv_simul}")
print(f"  sigrv_simul = {sigrv_simul}")

print()
print("Fit parameter summary:")
print(f"  RV        = {popt[0]:.3f} km/s")
print(f"  cont0     = {popt[1]:.3f}")
print(f"  slope     = {popt[2]:.5f}")
print(f"  CaT1 d,s,g = {popt[3]:.3f}, {popt[4]:.3f}, {popt[5]:.3f}")
print(f"  CaT2 d,s,g = {popt[6]:.3f}, {popt[7]:.3f}, {popt[8]:.3f}")
print(f"  CaT3 d,s,g = {popt[9]:.3f}, {popt[10]:.3f}, {popt[11]:.3f}")

In [ ]:
# ============================================================
# 10. Fit Voigt profiles to the three CaT lines
#     + residual panels
#     + robust RV combination
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

from scipy.optimize import curve_fit
from scipy.special import voigt_profile

# ------------------------------------------------------------
# Line information
# ------------------------------------------------------------

cat_names = ["CaT1 8498", "CaT2 8542", "CaT3 8662"]

rest_cat_AA = np.array([
    8498.02,
    8542.09,
    8662.14
], dtype=float)

c_kms = 299792.458

fit_half_width_AA = np.array([8.0, 10.0, 10.0], dtype=float)
ew_half_width_AA  = np.array([8.0, 10.0, 10.0], dtype=float)

rv_reject_threshold = 7.0  # km/s

# ------------------------------------------------------------
# Current normalized spectrum
# ------------------------------------------------------------

lam_AA = np.asarray(x_m, dtype=float) * 1.0e10
flux = np.asarray(normalized_flux_array, dtype=float)

good = (
    np.isfinite(lam_AA) &
    np.isfinite(flux) &
    (flux > 0)
)

lam_AA = lam_AA[good]
flux = flux[good]

# Initial CaT centers
picked_cat_AA = np.array([
    c1,
    c2,
    c3
], dtype=float) * 1.0e10

# ------------------------------------------------------------
# Voigt absorption model
# ------------------------------------------------------------

def voigt_absorption(
    lam,
    cont0,
    slope,
    depth,
    center,
    sigma,
    gamma
):

    continuum_local = cont0 + slope * (lam - center)

    profile = voigt_profile(
        lam - center,
        sigma,
        gamma
    )

    peak = voigt_profile(
        0.0,
        sigma,
        gamma
    )

    profile_norm = profile / peak

    return continuum_local * (
        1.0 - depth * profile_norm
    )

# ------------------------------------------------------------
# Storage arrays
# ------------------------------------------------------------

ew_results = []
rv_results = []
dl_results = []
center_results = []
fit_quality_results = []

# ------------------------------------------------------------
# Create figure
# ------------------------------------------------------------

fig_voigt, axes = plt.subplots(
    2,
    3,
    figsize=(21, 8),
    sharex="col",
    gridspec_kw={"height_ratios": [3.0, 1.0]}
)

# ------------------------------------------------------------
# Fit each CaT line
# ------------------------------------------------------------

for i, (
    name,
    c_guess,
    c_rest,
    fit_hw,
    ew_hw
) in enumerate(

    zip(
        cat_names,
        picked_cat_AA,
        rest_cat_AA,
        fit_half_width_AA,
        ew_half_width_AA
    )
):

    ax = axes[0, i]
    axr = axes[1, i]

    # --------------------------------------------------------
    # Local fit region
    # --------------------------------------------------------

    m = (
        np.isfinite(lam_AA) &
        np.isfinite(flux) &
        (lam_AA > c_guess - fit_hw) &
        (lam_AA < c_guess + fit_hw)
    )

    xfit = lam_AA[m]
    yfit = flux[m]

    if len(xfit) < 15:

        raise ValueError(
            f"Not enough points to fit {name}"
        )

    # --------------------------------------------------------
    # Initial guesses
    # --------------------------------------------------------

    cont0_guess = np.nanmedian(yfit)

    depth_guess = (
        cont0_guess - np.nanmin(yfit)
    ) / cont0_guess

    depth_guess = np.clip(
        depth_guess,
        0.05,
        0.95
    )

    p0 = [
        cont0_guess,
        0.0,
        depth_guess,
        c_guess,
        0.8,
        0.8
    ]

    lower = [
        0.4,
        -0.05,
        0.00,
        c_guess - 2.5,
        0.05,
        0.01
    ]

    upper = [
        1.4,
        0.05,
        0.98,
        c_guess + 2.5,
        4.00,
        6.00
    ]

    # --------------------------------------------------------
    # Voigt fit
    # --------------------------------------------------------

    try:

        popt, pcov = curve_fit(
            voigt_absorption,
            xfit,
            yfit,
            p0=p0,
            bounds=(lower, upper),
            maxfev=5000
        )

    except RuntimeError:

        print()
        print(
            f"WARNING: Voigt fit failed for {name}"
        )
        print()

        continue

    cont0, slope, depth, center_AA, sigma_AA, gamma_AA = popt

    # --------------------------------------------------------
    # Model curves
    # --------------------------------------------------------

    xplot = np.linspace(
        c_guess - fit_hw,
        c_guess + fit_hw,
        2000
    )

    ymodel = voigt_absorption(
        xplot,
        *popt
    )

    ycont = cont0 + slope * (
        xplot - center_AA
    )

    # --------------------------------------------------------
    # EW integration
    # --------------------------------------------------------

    xint = np.linspace(
        center_AA - ew_hw,
        center_AA + ew_hw,
        2000
    )

    yint = voigt_absorption(
        xint,
        *popt
    )

    cint = cont0 + slope * (
        xint - center_AA
    )

    ew_AA = np.trapz(
        1.0 - yint / cint,
        xint
    ) * u.AA

    # --------------------------------------------------------
    # RV
    # --------------------------------------------------------

    dl_AA = center_AA - c_rest

    rv = (
        (dl_AA / c_rest)
        * c_kms
        * u.km / u.s
    )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    ew_results.append(ew_AA)
    rv_results.append(rv)
    dl_results.append(dl_AA * 1.0e-10)

    center_results.append(center_AA)

    # --------------------------------------------------------
    # Residuals
    # --------------------------------------------------------

    ymodel_fit = voigt_absorption(
        xfit,
        *popt
    )

    resid = yfit - ymodel_fit

    rms_resid = np.nanstd(resid)

    fit_good = rms_resid < 0.08

    fit_quality_results.append(fit_good)

    quality_label = "OK" if fit_good else "CHECK"
    quality_color = (
        "darkgreen"
        if fit_good
        else "darkred"
    )

    # --------------------------------------------------------
    # Main panel
    # --------------------------------------------------------

    ax.plot(
        xfit,
        yfit,
        color="black",
        lw=1.1,
        label="Observed spectrum"
    )

    ax.plot(
        xplot,
        ymodel,
        color="blue",
        lw=2.3,
        label="Voigt fit"
    )

    ax.plot(
        xplot,
        ycont,
        color="red",
        ls="--",
        lw=1.5,
        label="Local continuum"
    )

    fill_mask = (
        (xplot >= center_AA - ew_hw) &
        (xplot <= center_AA + ew_hw)
    )

    ax.fill_between(
        xplot,
        ycont,
        ymodel,
        where=fill_mask,
        color="green",
        alpha=0.35,
        label="Voigt EW"
    )

    ax.axvline(
        c_rest,
        color="gray",
        ls="--",
        lw=1.5,
        label="Rest wavelength"
    )

    ax.axvline(
        c_guess,
        color="black",
        ls="--",
        lw=1.5,
        alpha=0.7,
        label="Initial center"
    )

    ax.axvline(
        center_AA,
        color="purple",
        ls=":",
        lw=2.0,
        label="Fitted center"
    )

    ax.set_title(
        name,
        fontsize=16
    )

    ax.set_xlim(
        c_guess - fit_hw,
        c_guess + fit_hw
    )

    ax.set_ylim(0.0, 1.25)

    if i == 0:

        ax.set_ylabel(
            "Normalized Flux",
            fontsize=14
        )

    ax.grid(True, alpha=0.35)

    ax.text(
        0.03,
        0.96,
        f"EW = {ew_AA.to(u.AA).value:.2f} Å\n"
        f"RV = {rv.to(u.km/u.s).value:.1f} km/s\n"
        f"RMS = {rms_resid:.3f}\n"
        f"{quality_label}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        color=quality_color,
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.8
        )
    )

    # --------------------------------------------------------
    # Residual panel
    # --------------------------------------------------------

    axr.axhline(
        0.0,
        color="gray",
        ls="--",
        lw=1.2
    )

    axr.plot(
        xfit,
        resid,
        color="black",
        lw=1.0
    )

    axr.set_xlim(
        c_guess - fit_hw,
        c_guess + fit_hw
    )

    axr.set_ylim(-0.12, 0.12)

    axr.grid(True, alpha=0.35)

    axr.set_xlabel(
        r"$\lambda$ ($\AA$)",
        fontsize=13
    )

    if i == 0:

        axr.set_ylabel(
            "Residual",
            fontsize=12
        )

    # --------------------------------------------------------
    # Print sanity checks
    # --------------------------------------------------------

    print(name)

    print(
        f"  Initial center = "
        f"{c_guess:.3f} Angstrom"
    )

    print(
        f"  Fitted center  = "
        f"{center_AA:.3f} Angstrom"
    )

    print(
        f"  Lab wavelength = "
        f"{c_rest:.3f} Angstrom"
    )

    print(
        f"  EW Voigt       = "
        f"{ew_AA.to(u.AA).value:.3f} Angstrom"
    )

    print(
        f"  RV_lab/raw     = "
        f"{rv.to(u.km/u.s).value:.2f} km / s"
    )

    print(
        f"  sigma          = "
        f"{sigma_AA:.3f} Angstrom"
    )

    print(
        f"  gamma          = "
        f"{gamma_AA:.3f} Angstrom"
    )

    print(
        f"  RMS residual   = "
        f"{rms_resid:.4f}"
    )

    print(
        f"  Fit quality    = "
        f"{quality_label}"
    )

    print()

# ------------------------------------------------------------
# Plot formatting
# ------------------------------------------------------------

handles, labels = axes[0, 0].get_legend_handles_labels()

fig_voigt.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=6,
    fontsize=9,
    frameon=True
)

fig_voigt.suptitle(
    "Voigt Fits to the Ca II Triplet",
    fontsize=22,
    y=1.05
)

plt.tight_layout(
    rect=[0, 0, 1, 0.94]
)

plt.show()

# ------------------------------------------------------------
# Save fitted centers
# ------------------------------------------------------------

fitted_cat_AA = np.array(
    center_results,
    dtype=float
)

picked_cat = fitted_cat_AA * 1.0e-10

c1, c2, c3 = picked_cat

cat1, cat2, cat3 = ew_results

rv1_lab, rv2_lab, rv3_lab = rv_results

dl1, dl2, dl3 = dl_results

SEW = (
    cat1 + cat2 + cat3
).to(u.AA)

SEW2 = (
    cat2 + cat3
).to(u.AA)

# ------------------------------------------------------------
# Robust RV combination
# ------------------------------------------------------------

rv_values = np.array([
    rv.to(u.km/u.s).value
    for rv in rv_results
], dtype=float)

rv_names = np.array(cat_names)

rv_median = np.median(rv_values)

rv_diff = np.abs(
    rv_values - rv_median
)

worst_index = np.argmax(rv_diff)

use_mask = np.ones(
    3,
    dtype=bool
)

if rv_diff[worst_index] > rv_reject_threshold:

    use_mask[worst_index] = False

    print()
    print("WARNING:")
    print(
        f"Rejecting "
        f"{rv_names[worst_index]} "
        f"from final RV."
    )

    print(
        f"Discrepant RV = "
        f"{rv_values[worst_index]:.2f} km/s"
    )

    print(
        f"Median RV     = "
        f"{rv_median:.2f} km/s"
    )

    print(
        f"|Delta RV|    = "
        f"{rv_diff[worst_index]:.2f} km/s"
    )

    print()

rv_values_used = rv_values[use_mask]

# ------------------------------------------------------------
# Final Voigt RV
# ------------------------------------------------------------

avrv = np.mean(
    rv_values_used
) * u.km / u.s

if len(rv_values_used) > 1:

    sigrv = np.std(
        rv_values_used,
        ddof=1
    ) * u.km / u.s

else:

    sigrv = 0.0 * u.km / u.s

# ------------------------------------------------------------
# Save Voigt RV solution
# ------------------------------------------------------------

avrv_voigt = avrv
sigrv_voigt = sigrv

print()
print("Saved Voigt RV solution:")

print(
    f"  avrv_voigt  = "
    f"{avrv_voigt:.3f}"
)

print(
    f"  sigrv_voigt = "
    f"{sigrv_voigt:.3f}"
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()
print("RV lines used:")

for name, rv, keep in zip(
    rv_names,
    rv_values,
    use_mask
):

    status = (
        "USED"
        if keep
        else "REJECTED"
    )

    print(
        f"  {name:10s} "
        f"{rv:8.2f} km/s   "
        f"{status}"
    )

print()
print("Summary:")

for name, ew, rv, keep in zip(
    cat_names,
    ew_results,
    rv_results,
    use_mask
):

    status = (
        "USED"
        if keep
        else "RV REJECTED"
    )

    print(
        f"{name}: "
        f"EW = {ew.to(u.AA).value:.3f} Angstrom, "
        f"RV = {rv.to(u.km/u.s).value:.2f} km / s, "
        f"{status}"
    )

print()

print(
    f"Total CaT EW = "
    f"{SEW.to(u.AA).value:.3f} Angstrom"
)

print(
    f"CaT2 + CaT3 EW = "
    f"{SEW2.to(u.AA).value:.3f} Angstrom"
)

print(
    f"Adopted Mean RV_lab/raw = "
    f"{avrv.to(u.km/u.s).value:.3f} "
    f"+/- "
    f"{sigrv.to(u.km/u.s).value:.3f} km / s"
)

print()
print(
    "These Voigt-fitted centers now define c1, c2, c3."
)

print(
    "The adopted avrv uses only the RV lines marked USED."
)

In [ ]:
# ============================================================
# Final RV sanity check: simultaneous RV vs Voigt RVs
# ============================================================

rv_flag = "GOOD"

if "avrv_simul" in globals():

    rv_simul = avrv_simul.to(u.km/u.s).value
    rv_voigt = avrv.to(u.km/u.s).value

    delta_rv = abs(rv_voigt - rv_simul)

    if delta_rv > 10.0:
        rv_flag = "BAD_RV_DISAGREEMENT"

        print("WARNING: BAD RV SOLUTION")
        print(f"Simultaneous RV = {rv_simul:.2f} km/s")
        print(f"Voigt RV        = {rv_voigt:.2f} km/s")
        print(f"Difference      = {delta_rv:.2f} km/s")
        print()
        print("This means the Voigt fits converged mathematically,")
        print("but the simultaneous fits are probably fitting the wrong feature or a distorted profile.")
        print("Do NOT trust this RV without manual inspection.")

    else:
        print("RV solutions agree.")
        print(f"Simultaneous RV = {rv_simul:.2f} km/s")
        print(f"Voigt RV        = {rv_voigt:.2f} km/s")
        print(f"Difference      = {delta_rv:.2f} km/s")

else:
    rv_flag = "NO_SIMULTANEOUS_RV"
    print("No simultaneous RV available for comparison.")

print(f"RV flag = {rv_flag}")

In [ ]:
# ============================================================
# Choose final RV solution before shifting spectrum
# ============================================================
#LOOK AT THE FITS: THERE ARE SOME bad continuum / blended cool-star spectrum / sky residual contamination / possible composite or non-member STARS-FLAG THEM!
# This cell lets you decide which RV solution to adopt:
#
#   v = individual Voigt-line RV
#   s = simultaneous shared-RV fit
#
# The chosen RV becomes:
#
#   avrv
#   sigrv
#
# which are then used for:
#   - rest-frame wavelength shift
#   - SP_Ace export
#   - final output table
# ============================================================

import astropy.units as u

print()
print("Available RV solutions")
print("----------------------")

# ------------------------------------------------------------
# Voigt RV
# ------------------------------------------------------------

if "avrv_voigt" in globals():

    print(
        f"Voigt-line RV       = "
        f"{avrv_voigt.to(u.km/u.s).value:.3f} "
        f"+/- "
        f"{sigrv_voigt.to(u.km/u.s).value:.3f} km/s"
    )

else:

    print("Voigt-line RV       = NOT AVAILABLE")

# ------------------------------------------------------------
# Simultaneous RV
# ------------------------------------------------------------

if "avrv_simul" in globals():

    print(
        f"Simultaneous CaT RV = "
        f"{avrv_simul.to(u.km/u.s).value:.3f} "
        f"+/- "
        f"{sigrv_simul.to(u.km/u.s).value:.3f} km/s"
    )

else:

    print("Simultaneous CaT RV = NOT AVAILABLE")

print()
print("Choose RV solution:")
print("  v = use individual Voigt-line RV-PREFERRED IF IT WORKS!")
print("  s = use simultaneous shared-RV fit-USE IF ONE OF THE CaT LINES IS REJECTED OR IF ONE OF THE LINES IS SHALLOW OR BROAD!")
print()

choice = input("Use which RV solution? (v/s): ").strip().lower()

# ------------------------------------------------------------
# Use Voigt RV
# ------------------------------------------------------------

if choice in ["v", "voigt"]:

    if "avrv_voigt" not in globals():

        raise ValueError(
            "Voigt RV solution does not exist."
        )

    avrv = avrv_voigt
    sigrv = sigrv_voigt

    rv_method_used = "VOIGT"

# ------------------------------------------------------------
# Use simultaneous RV
# ------------------------------------------------------------

elif choice in ["s", "simul", "simultaneous"]:

    if "avrv_simul" not in globals():

        raise ValueError(
            "Simultaneous RV solution does not exist."
        )

    avrv = avrv_simul
    sigrv = sigrv_simul

    rv_method_used = "SIMULTANEOUS"

# ------------------------------------------------------------
# Invalid choice
# ------------------------------------------------------------

else:

    raise ValueError(
        "Choice not recognized. Enter 'v' or 's'."
    )

# ------------------------------------------------------------
# Final adopted RV
# ------------------------------------------------------------

print()
print("Final adopted RV solution")
print("-------------------------")

print(f"Method used = {rv_method_used}")

print(
    f"RV_lab/raw  = "
    f"{avrv.to(u.km/u.s).value:.3f} "
    f"+/- "
    f"{sigrv.to(u.km/u.s).value:.3f} km/s"
)

print()
print("This RV will now be used for:")
print("  - rest-frame wavelength shift")
print("  - SP_Ace export")
print("  - final output table")

In [ ]:
# ============================================================
# 11. RV sanity checks and reporting-frame corrections
# ============================================================
# avrv is the lab-frame/raw RV from the Voigt fitted CaT centers.
# This is the RV used to shift the spectrum into the stellar rest/lab frame for SP_Ace.
# The barycentric/APOGEE values are for reporting/comparison only.

# Change this if the barycentric correction differs for a different observation.
barycorr = -13.353688041694257 * u.km/u.s

RV_barycentric = avrv + barycorr

# Empirical correction to APOGEE system from standards.
RV_apogee_fit = (0.999 * avrv.to_value(u.km/u.s) - 13.758) * u.km/u.s

# Error model from line-to-line scatter plus standards-fit scatter.
rverr = np.sqrt(sigrv.to_value(u.km/u.s)**2 + 3.204**2) * u.km/u.s

print("Individual CaT equivalent widths")
print("EW CaT1\t\tEW CaT2\t\tEW CaT3")
print(f"{cat1.to(u.AA):.3f}\t\t{cat2.to(u.AA):.3f}\t\t{cat3.to(u.AA):.3f}")

print("\nSummed CaT EWs")
print(f"Sum of 3 CaT EW = {SEW:.5f}")
print(f"Sum of 2 CaT EW = {SEW2:.5f}")

print("\nVoigt fitted CaT wavelengths, m")
print(f"{c1:.9e}\t{c2:.9e}\t{c3:.9e}")

print("\nRaw/lab-frame CaT RVs")
print(f"{rv1_lab:.3f}\t{rv2_lab:.3f}\t{rv3_lab:.3f}")

print("\nRadial velocities")
print(f"RV_lab/raw = {avrv:.3f} +/- {sigrv:.3f}")
print(f"Barycentric correction applied = {barycorr:.3f}")
print(f"RV_barycentric = {RV_barycentric:.3f} +/- {sigrv:.3f}")
print(f"RV_APOGEE_fit = {RV_apogee_fit:.3f} +/- {rverr:.3f}")

# Simple automated quality flags.
rv_line_values = np.array([rv1_lab.to_value(u.km/u.s), rv2_lab.to_value(u.km/u.s), rv3_lab.to_value(u.km/u.s)])
ew_values = np.array([cat1.to_value(u.AA), cat2.to_value(u.AA), cat3.to_value(u.AA)])

print("\nSanity checks")
print("-------------")
print(f"RV line-to-line scatter = {sigrv:.3f}")
print(f"EW ordering CaT2 > CaT3 > CaT1? {ew_values[1] > ew_values[2] > ew_values[0]}")
print(f"CaT1 positive? {ew_values[0] > 0}")
print(f"CaT2 positive? {ew_values[1] > 0}")
print(f"CaT3 positive? {ew_values[2] > 0}")

if sigrv.to_value(u.km/u.s) > 10:
    print("WARNING: RV scatter is large. Inspect the Voigt fits and consider rejecting a line.")

if not (ew_values[1] > ew_values[0] and ew_values[2] > ew_values[0]):
    print("WARNING: CaT EW ordering is unusual. Inspect continuum and line fits.")


In [ ]:
# ============================================================
# 12. Shift normalized spectrum to stellar rest/lab frame
#     and check SP_Ace wavelength alignment
# ============================================================
# SP_Ace input should be:
#
#   column 1 = wavelength in Angstroms
#              shifted to stellar rest/lab frame
#
#   column 2 = continuum-normalized flux
#
# Uses the FINAL adopted RV:
#   avrv
#
# which was chosen in the RV-selection cell.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

# ------------------------------------------------------------
# Speed of light
# ------------------------------------------------------------

c_kms = 299792.458

# ------------------------------------------------------------
# Final adopted RV
# ------------------------------------------------------------

rv_shift = avrv.to_value(u.km/u.s)

beta = rv_shift / c_kms

print()
print("SP_Ace rest-frame wavelength shift")
print("----------------------------------")

print(f"RV method used = {rv_method_used}")

print(
    f"RV used for rest-frame shift = "
    f"{rv_shift:.3f} km/s"
)

# ------------------------------------------------------------
# Input normalized spectrum
# ------------------------------------------------------------

wvl_obs_m = np.asarray(
    x_m,
    dtype=float
)

flux_for_space = np.asarray(
    normalized_flux_array,
    dtype=float
)

good = (
    np.isfinite(wvl_obs_m) &
    np.isfinite(flux_for_space) &
    (flux_for_space > 0)
)

wvl_obs_m = wvl_obs_m[good]
flux_for_space = flux_for_space[good]

# ------------------------------------------------------------
# Sort spectrum
# ------------------------------------------------------------

order = np.argsort(wvl_obs_m)

wvl_obs_m = wvl_obs_m[order]
flux_for_space = flux_for_space[order]

# ------------------------------------------------------------
# Shift to stellar rest/lab frame
# ------------------------------------------------------------
# Positive RV means observed wavelengths are redshifted.
#
# Remove the Doppler shift by dividing by:
#
#   (1 + v/c)
# ------------------------------------------------------------

wvl_rest_m = wvl_obs_m / (1.0 + beta)

wvl_rest_AA = wvl_rest_m * 1.0e10

# ------------------------------------------------------------
# Lab CaT wavelengths
# ------------------------------------------------------------

rest_cat_AA = np.array([
    8498.02,
    8542.09,
    8662.14
])

rest_cat_m = rest_cat_AA * 1.0e-10

# ------------------------------------------------------------
# Major spectral features in CaT region
# ------------------------------------------------------------

major_lines = [

    # Ca II triplet
    (8498.02, "Ca II\n8498"),
    (8542.09, "Ca II\n8542"),
    (8662.14, "Ca II\n8662"),

    # Hydrogen Paschen lines
    (8502.49, "H I P16\n8502"),
    (8545.38, "H I P15\n8545"),
    (8598.39, "H I P14\n8598"),
    (8665.02, "H I P13\n8665"),
    (8750.47, "H I P12\n8750"),

    # Metal lines
    (8514.07, "Fe I\n8514"),
    (8526.67, "Fe I\n8527"),
    (8582.27, "Fe I\n8582"),
    (8611.80, "Fe I\n8612"),
    (8674.75, "Fe I\n8675"),
    (8688.63, "Fe I\n8689"),
    (8806.76, "Mg I\n8807"),
]

major_lines = sorted(
    major_lines,
    key=lambda x: x[0]
)

# ------------------------------------------------------------
# Label staggering
# ------------------------------------------------------------

label_levels = [
    1.44,
    1.36,
    1.28,
    1.20
]

x_offsets_AA = [
    -0.30,
     0.30,
    -0.15,
     0.15
]

# ------------------------------------------------------------
# Plot wavelength check
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(18, 5.5)
)

# Observed wavelength grid
ax.plot(
    wvl_obs_m,
    flux_for_space,
    color="green",
    lw=1,
    alpha=0.35,
    label="Observed normalized"
)

# Shifted rest-frame grid
ax.plot(
    wvl_rest_m,
    flux_for_space,
    color="blue",
    lw=1,
    label="Rest-frame wavelength grid"
)

# ------------------------------------------------------------
# Voigt fitted centers BEFORE shift
# ------------------------------------------------------------

for i, c in enumerate([c1, c2, c3]):

    ax.axvline(
        c,
        ls=":",
        color="black",
        alpha=0.65,
        label="Voigt center" if i == 0 else None
    )

# ------------------------------------------------------------
# Lab CaT wavelengths
# ------------------------------------------------------------

for i, rc in enumerate(rest_cat_m):

    ax.axvline(
        rc,
        ls=":",
        color="red",
        alpha=0.9,
        linewidth=1.2,
        label="Lab CaT" if i == 0 else None
    )

# ------------------------------------------------------------
# Other major line labels
# ------------------------------------------------------------

for j, (lam_AA, label) in enumerate(major_lines):

    lam_m = lam_AA * 1.0e-10

    ypos = label_levels[
        j % len(label_levels)
    ]

    dx_AA = x_offsets_AA[
        j % len(x_offsets_AA)
    ]

    ax.axvline(
        lam_m,
        color="purple",
        linestyle=":",
        alpha=0.28,
        linewidth=0.9
    )

    ax.text(
        (lam_AA + dx_AA) * 1.0e-10,
        ypos,
        label,
        rotation=90,
        ha="center",
        va="top",
        fontsize=7,
        color="purple",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.65,
            pad=0.25
        )
    )

# ------------------------------------------------------------
# Plot formatting
# ------------------------------------------------------------

ax.set_xlim(
    8.47e-7,
    8.85e-7
)

ax.set_ylim(
    0,
    1.5
)

ax.set_xlabel(
    "Wavelength (m)"
)

ax.set_ylabel(
    "Normalized Flux"
)

ax.set_title(
    "SP_Ace Rest-Frame Wavelength Check"
)

ax.grid(
    True,
    alpha=0.35
)

ax.legend(
    loc="lower right",
    fontsize=10,
    frameon=True
)

plt.tight_layout()

plt.show()

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

print()
print(
    "After the shift, the BLUE CaT lines "
    "should line up with the RED lab wavelengths."
)

print()
print("Rest-frame CaT wavelengths:")

for rc in rest_cat_AA:

    print(f"  {rc:.2f} Angstrom")

print()
print("Output arrays available for SP_Ace export:")

print(
    "  wvl_rest_AA    = rest-frame wavelength (Angstrom)"
)

print(
    "  flux_for_space = normalized flux"
)

print()
print(
    f"Rest-frame wavelength range = "
    f"{np.nanmin(wvl_rest_AA):.2f} "
    f"to "
    f"{np.nanmax(wvl_rest_AA):.2f} Angstrom"
)

In [ ]:
#
# ============================================================
# 13. Export normalized rest-frame spectrum for SP_Ace
# ============================================================
# Output:
#   column 1 = wavelength (Angstrom)
#   column 2 = normalized flux
#
# Only export the scientifically usable wavelength range.
# ============================================================

from astropy.table import Table
from astropy.io import ascii
from pathlib import Path
import numpy as np

# ------------------------------------------------------------
# Safe wavelength limits for AAT CaT spectra
# ------------------------------------------------------------

wave_min_AA = 8450.0
wave_max_AA = 8850.0

# ------------------------------------------------------------
# Build export mask
# ------------------------------------------------------------

export_mask = (
    np.isfinite(wvl_rest_AA) &
    np.isfinite(flux_for_space) &
    (flux_for_space > 0) &
    (wvl_rest_AA >= wave_min_AA) &
    (wvl_rest_AA <= wave_max_AA)
)

wave_export = wvl_rest_AA[export_mask]
flux_export = flux_for_space[export_mask]

# ------------------------------------------------------------
# Final sort
# ------------------------------------------------------------

order = np.argsort(wave_export)

wave_export = wave_export[order]
flux_export = flux_export[order]

# ------------------------------------------------------------
# Create SP_Ace table
# ------------------------------------------------------------

data = Table()

data["x"] = np.round(wave_export, 4)
data["y"] = np.round(flux_export, 6)


# ------------------------------------------------------------
# Output filename for SP_Ace
# ------------------------------------------------------------

out_dir = Path(os.getenv("NORMALIZED_AAT_DIR", PROJECT_ROOT / "data/interim/aat_normalized_lab_frame")).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
outfile = out_dir / f"aat_{file_number}.txt"

# ------------------------------------------------------------
# Write file
# ------------------------------------------------------------

ascii.write(
    data,
    outfile,
    format="no_header",
    delimiter=" ",
    overwrite=True
)

# ------------------------------------------------------------
# Plot exported region
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(16,5))

ax.plot(
    wave_export,
    flux_export,
    lw=1,
    color="blue"
)

ax.axhline(
    1.0,
    color="black",
    ls="--",
    alpha=0.5
)

ax.set_xlabel("Wavelength (Å)")
ax.set_ylabel("Normalized Flux")

ax.set_title(
    f"SP_Ace Export Spectrum: Fiber {fiberid}"
)

ax.set_ylim(0, 1.5)

ax.grid(True)

plt.show()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("SP_Ace export complete.")
print()

print(f"Fiber ID           = {fiberid}")
print(f"Output file        = {outfile}")

print()
print("Export wavelength range:")
print(f"  {wave_export[0]:.2f} Å  to  {wave_export[-1]:.2f} Å")

print()
print(f"Number of exported pixels = {len(wave_export)}")

print()
print("RV used for rest-frame shift:")
print(f"  {rv_shift:.3f} km/s")

print()
print("This exported spectrum is:")
print("  - continuum normalized")
print("  - shifted to stellar rest/lab frame")
print("  - trimmed to the stable CaT wavelength region")


In [ ]:
# ============================================================
# 14. Append compact results summary table
# ============================================================
# This appends one row to the results table and does not overwrite previous stars.

import os
from pathlib import Path
import astropy.units as u

# ------------------------------------------------------------
# Results file
# ------------------------------------------------------------
results_file = str(Path(os.getenv("INTERIM_DATA_DIR", PROJECT_ROOT / "data/interim")).resolve() / "CaT_summary_table.txt")

# ------------------------------------------------------------
# Rebuild fiber ID from spec_path if needed
# ------------------------------------------------------------
try:
    fiberid
except NameError:
    fname = Path(spec_path).stem
    fiberid = fname.split("_")[-1]

# ------------------------------------------------------------
# Helper to safely convert quantities to float values
# ------------------------------------------------------------
def qvalue(x, unit=None):
    if unit is not None and hasattr(x, "to_value"):
        return x.to_value(unit)
    if hasattr(x, "value"):
        return float(x.value)
    return float(x)

# ------------------------------------------------------------
# Check that required values exist before writing
# ------------------------------------------------------------
required_names = [
    "fiberid", "avrv", "RV_barycentric", "rverr", "snr",
    "cat1", "cat2", "cat3", "SEW2"
]

missing = [name for name in required_names if name not in globals()]

if missing:
    raise NameError(
        "Cannot write results table because these variables are missing: "
        + ", ".join(missing)
    )

# ------------------------------------------------------------
# Create directory and header if needed
# ------------------------------------------------------------
outdir = os.path.dirname(results_file)
if outdir:
    os.makedirs(outdir, exist_ok=True)

header = "fiberid\tRVraw\tRV\te_RV\tS/N\tCaT1\tCaT2\tCaT3\tSumEW23\n"

if (not os.path.exists(results_file)) or os.path.getsize(results_file) == 0:
    with open(results_file, "w", newline="") as f:
        f.write(header)

# ------------------------------------------------------------
# Build row
# ------------------------------------------------------------
row = (
    f"{fiberid}\t"
    f"{qvalue(avrv, u.km/u.s):.3f}\t"
    f"{qvalue(RV_barycentric, u.km/u.s):.3f}\t"
    f"{qvalue(rverr, u.km/u.s):.3f}\t"
    f"{qvalue(snr):.1f}\t"
    f"{qvalue(cat1, u.AA):.3f}\t"
    f"{qvalue(cat2, u.AA):.3f}\t"
    f"{qvalue(cat3, u.AA):.3f}\t"
    f"{qvalue(SEW2, u.AA):.3f}\n"
)

# ------------------------------------------------------------
# Append row
# ------------------------------------------------------------
with open(results_file, "a", newline="") as f:
    f.write(row)
    f.flush()
    os.fsync(f.fileno())

print(f"Added star {fiberid} to:")
print(results_file)
print("\nRow written:")
print(header.strip())
print(row.strip())

# ------------------------------------------------------------
# Read back last few lines as a sanity check
# ------------------------------------------------------------
print("\nLast lines now in results file:")
with open(results_file, "r") as f:
    lines = f.readlines()

for line in lines[-5:]:
    print(line.rstrip())



## End of one-star workflow

For the next star:
1. Go back to **Cell 1** and clear old variables.
2. Choose the next FITS file in **Cell 2**.
3. Repeat the workflow.

Remember: if the continuum fit is bad, do not force it. Use the restricted CaT-only normalization and flag the star if the Voigt fits are not physically plausible.